# 🏡 Home Price Prediction System Using Machine Learning
### An End-to-End Predictive Modeling and Valuation Pipeline

**Author:** AI/ML Engineering Team  
**Dataset:** Metropolitan Real Estate Sales (5,000 Records, 17 Attributes)  
**Algorithms:** Linear Regression, Ridge, Lasso, Decision Trees, Random Forest, Gradient Boosting, XGBoost Regressor  
**Evaluation Metrics:** $R^2$, MAE, MSE, RMSE, MAPE  

---
## Project Overview & Objectives
Real estate pricing is driven by complex interactions among location desirability, square footage, structural age, architectural grade, and luxury amenities. The goal of this project is to build, compare, and fine-tune multiple machine learning models to accurately predict home prices, identify key valuation drivers, and deploy an automated valuation engine.


## 1. Environment Setup & Library Ingestion
Importing core computational, statistical, visualization, and machine learning libraries.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and modeling
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import xgboost as xgb
import joblib

# Plotting configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'


## 2. Dataset Ingestion & Exploratory Inspection
We load the real estate transaction dataset containing features like location, area (sqft), bedrooms, bathrooms, year built, amenities, and historical sales prices.


In [ ]:
# Load real estate dataset
data_path = os.path.join("..", "data", "house_prices.csv")
if not os.path.exists(data_path):
    data_path = "house_prices.csv"

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
display(df.head())


In [ ]:
# Dataset summary and data types
print("Dataset Information:")
df.info()

print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nDescriptive Statistics for Numerical Features:")
display(df.describe().round(2))


## 3. Exploratory Data Analysis (EDA)
In this phase, we analyze:
1. Target price distribution and skewness
2. House prices across different neighborhoods/locations
3. Living space (sqft) vs Price relationships colored by quality grade
4. Feature correlation matrix


In [ ]:
# 3.1 Target Variable (Price) Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['price'] / 1000, kde=True, ax=ax1, color="#2b5c8f", bins=30)
ax1.set_title("Property Price Distribution ($1,000s)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Price ($ Thousands)")

sns.boxplot(x=df['price'] / 1000, ax=ax2, color="#48cae4")
ax2.set_title("Price Boxplot (Spread & Outlier Check)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Price ($ Thousands)")

plt.tight_layout()
plt.show()

print(f"Price Skewness: {df['price'].skew():.2f}")
print(f"Price Kurtosis: {df['price'].kurtosis():.2f}")


In [ ]:
# 3.2 Price Variance Across Neighborhoods
plt.figure(figsize=(12, 6))
order = df.groupby("location")["price"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="price", y="location", order=order, palette="crest", orient="h")
plt.title("Property Price Distribution by Neighborhood", fontsize=13, fontweight='bold')
plt.xlabel("Price ($)")
plt.gca().xaxis.set_major_formatter("${x:,.0f}")
plt.show()


In [ ]:
# 3.3 Living Area vs. Price colored by Construction Grade
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=df,
    x="sqft_living",
    y="price",
    hue="grade",
    size="bathrooms",
    sizes=(30, 200),
    palette="viridis",
    alpha=0.65
)
plt.title("Living Area vs. Price (Colored by Architectural Grade)", fontsize=13, fontweight='bold')
plt.xlabel("Living Area (Square Feet)")
plt.ylabel("Price ($)")
plt.gca().yaxis.set_major_formatter("${x:,.0f}")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# 3.4 Correlation Matrix
plt.figure(figsize=(12, 9))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Matrix with House Price", fontsize=13, fontweight='bold')
plt.show()


## 4. Feature Engineering
Feature engineering leverages domain expertise to build predictive features:
- **`property_age`**: Chronological property age ($2025 - \text{yr\_built}$)
- **`effective_age`**: Effective age considering renovation date
- **`is_renovated`**: Binary indicator of whether property underwent renovations
- **`total_rooms`**: Aggregate room count (Bedrooms + Bathrooms)
- **`sqft_per_room`**: Spatial spaciousness per room
- **`living_to_lot_ratio`**: Proportion of parcel occupied by structure
- **`luxury_score`**: Weighted composite score incorporating waterfront, scenic view quality, and swimming pool
- **`condition_grade_interaction`**: Joint interaction term between structural grade and physical condition


In [ ]:
def engineer_features(data: pd.DataFrame, current_year: int = 2025) -> pd.DataFrame:
    df_feat = data.copy()
    
    # 1. Age calculations
    df_feat["property_age"] = np.clip(current_year - df_feat["yr_built"], 0, None)
    df_feat["is_renovated"] = (df_feat["yr_renovated"] > 0).astype(int)
    df_feat["effective_age"] = np.where(
        df_feat["yr_renovated"] > 0,
        np.clip(current_year - df_feat["yr_renovated"], 0, None),
        df_feat["property_age"]
    )
    df_feat["years_since_renovation"] = np.where(
        df_feat["yr_renovated"] > 0,
        np.clip(current_year - df_feat["yr_renovated"], 0, None),
        0
    )
    
    # 2. Room density metrics
    df_feat["total_rooms"] = df_feat["bedrooms"] + df_feat["bathrooms"]
    df_feat["bed_bath_ratio"] = np.where(df_feat["bathrooms"] > 0, df_feat["bedrooms"] / df_feat["bathrooms"], df_feat["bedrooms"])
    df_feat["sqft_per_room"] = df_feat["sqft_living"] / df_feat["total_rooms"]
    df_feat["living_to_lot_ratio"] = df_feat["sqft_living"] / df_feat["sqft_lot"]
    df_feat["unbuilt_lot_area"] = np.clip(df_feat["sqft_lot"] - df_feat["sqft_living"], 0, None)
    
    # 3. Composite luxury score
    df_feat["luxury_score"] = (
        df_feat["waterfront"] * 3.0 +
        df_feat["view_quality"] * 1.5 +
        df_feat["has_pool"] * 1.5 +
        np.where(df_feat["grade"] >= 9, (df_feat["grade"] - 8) * 1.2, 0.0)
    )
    
    # 4. Interaction term
    df_feat["condition_grade_interaction"] = df_feat["condition"] * df_feat["grade"]
    
    return df_feat

df_engineered = engineer_features(df)
print(f"Features after engineering: {df_engineered.shape[1]} columns")
display(df_engineered[["property_age", "effective_age", "total_rooms", "luxury_score", "condition_grade_interaction"]].head())


## 5. Data Preprocessing & Column Transformation Pipeline
We isolate the target variable (`price`) and partition the data into training (80%) and testing (20%) sets. A scikit-learn `ColumnTransformer` handles:
- **Numerical Features**: Scaled using `StandardScaler` (zero mean, unit variance)
- **Categorical Features**: One-hot encoded using `OneHotEncoder(handle_unknown='ignore')`


In [ ]:
# Define feature schemas
categorical_features = ["location"]
numerical_features = [
    "sqft_living", "bedrooms", "bathrooms", "sqft_lot", "floors",
    "waterfront", "view_quality", "condition", "grade",
    "distance_to_center_miles", "has_garage", "has_pool",
    "property_age", "effective_age", "is_renovated", "years_since_renovation",
    "total_rooms", "bed_bath_ratio", "sqft_per_room", "living_to_lot_ratio",
    "unbuilt_lot_area", "luxury_score", "condition_grade_interaction"
]
target = "price"

feature_cols = categorical_features + numerical_features
X = df_engineered[feature_cols]
y = df_engineered[target]

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build preprocessing ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Reconstruct DataFrame with feature names
cat_cols = list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features))
all_features = numerical_features + cat_cols

X_train_df = pd.DataFrame(X_train_trans, columns=all_features, index=X_train.index)
X_test_df = pd.DataFrame(X_test_trans, columns=all_features, index=X_test.index)

print(f"X_train shape: {X_train_df.shape} | X_test shape: {X_test_df.shape}")


## 6. Model Training & 5-Fold Cross-Validation
We train and benchmark 7 distinct regression algorithms:
1. **Linear Regression**: Classical Ordinary Least Squares
2. **Ridge Regression**: L2-regularized linear model
3. **Lasso Regression**: L1-regularized sparse feature selector
4. **Decision Tree Regressor**: Non-linear tree splitting
5. **Random Forest Regressor**: Bagging ensemble of de-correlated trees
6. **Gradient Boosting Regressor**: Sequential residual boosting
7. **XGBoost Regressor**: Optimized gradient boosting with second-order Taylor expansion


In [ ]:
candidate_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0, random_state=42),
    "Lasso Regression": Lasso(alpha=50.0, max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=14, min_samples_split=6, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=200, learning_rate=0.06, max_depth=6, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in candidate_models.items():
    cv_r2 = cross_val_score(model, X_train_df, y_train, cv=kf, scoring="r2", n_jobs=-1)
    cv_neg_rmse = cross_val_score(model, X_train_df, y_train, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1)
    model.fit(X_train_df, y_train)
    
    cv_results[name] = {
        "model": model,
        "cv_r2_mean": float(np.mean(cv_r2)),
        "cv_r2_std": float(np.std(cv_r2)),
        "cv_rmse": float(-np.mean(cv_neg_rmse))
    }
    print(f"{name:<20}: CV R² = {cv_results[name]['cv_r2_mean']:.4f} (+/- {cv_results[name]['cv_r2_std']:.4f}) | CV RMSE = ${cv_results[name]['cv_rmse']:,.0f}")


## 7. Hyperparameter Tuning (XGBoost Regressor)
We use `GridSearchCV` to optimize the tree depth, learning rate, subsample ratio, and tree estimators of XGBoost.


In [ ]:
param_grid = {
    "n_estimators": [150, 250],
    "learning_rate": [0.05, 0.08],
    "max_depth": [5, 7],
    "subsample": [0.8, 0.95],
    "colsample_bytree": [0.8, 0.95]
}

grid_search = GridSearchCV(
    estimator=xgb.XGBRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)
grid_search.fit(X_train_df, y_train)
best_xgb = grid_search.best_estimator_

print(f"Optimal Hyperparameters: {grid_search.best_params_}")


## 8. Comparative Evaluation on Holdout Test Set
We calculate:
- **Mean Absolute Error (MAE)**: Average absolute magnitude of errors in Rupees (₹)
- **Mean Squared Error (MSE)**: Squared penalty on deviations
- **Root Mean Squared Error (RMSE)**: Interpretable standard deviation of prediction residuals
- **R-squared ($R^2$)**: Proportion of price variance explained
- **Mean Absolute Percentage Error (MAPE)**: Percentage error relative to actual property price


In [ ]:
# Evaluate all models
eval_records = []
all_models = {**candidate_models, "XGBoost (Tuned)": best_xgb}

for name, model in all_models.items():
    preds = model.predict(X_test_df)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds) * 100
    
    eval_records.append({
        "Model": name,
        "Test R² Score": round(r2, 4),
        "Test MAE ($)": round(mae, 2),
        "Test RMSE ($)": round(rmse, 2),
        "MAPE (%)": round(mape, 2)
    })

eval_df = pd.DataFrame(eval_records).sort_values(by="Test R² Score", ascending=False)
display(eval_df)


In [ ]:
# Visualization of Evaluation Benchmarks
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=eval_df, x="Test R² Score", y="Model", palette="crest", ax=ax1)
ax1.set_title("Model Comparison: Test R² Score (Higher is Better)", fontsize=12, fontweight='bold')
ax1.set_xlim(0.8, 1.0)
for i, v in enumerate(eval_df["Test R² Score"]):
    ax1.text(v + 0.002, i, f"{v:.4f}", va="center", fontweight="bold")

sns.barplot(data=eval_df, x="Test RMSE ($)", y="Model", palette="flare", ax=ax2)
ax2.set_title("Model Comparison: Test RMSE ($) (Lower is Better)", fontsize=12, fontweight='bold')
ax2.xaxis.set_major_formatter("${x:,.0f}")
for i, v in enumerate(eval_df["Test RMSE ($)"]):
    ax2.text(v + 1500, i, f"${v:,.0f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()


## 9. Diagnostic Residual Analysis
We inspect the residuals ($y - \hat{y}$) for the winning Tuned XGBoost model to ensure normality and lack of heteroscedasticity.


In [ ]:
best_predictions = best_xgb.predict(X_test_df)
residuals = y_test - best_predictions

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
ax1.scatter(y_test, best_predictions, alpha=0.45, color="#1e3a8a")
min_val = min(y_test.min(), best_predictions.min())
max_val = max(y_test.max(), best_predictions.max())
ax1.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Perfect Agreement (y = x)")
ax1.set_title("Tuned XGBoost: Actual vs. Predicted Prices", fontsize=12, fontweight='bold')
ax1.set_xlabel("Actual Price ($)")
ax1.set_ylabel("Predicted Price ($)")
ax1.xaxis.set_major_formatter("${x:,.0f}")
ax1.yaxis.set_major_formatter("${x:,.0f}")
ax1.legend()

# Residual Distribution
sns.histplot(residuals, kde=True, ax=ax2, color="#059669", bins=30)
ax2.axvline(0, color="black", linestyle="--", lw=1.5)
ax2.set_title("Residual Error Distribution (Actual - Predicted)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Residual ($)")
ax2.xaxis.set_major_formatter("${x:,.0f}")

plt.tight_layout()
plt.show()


## 10. Feature Importance & Model Interpretability
Extracting the top predictive features driving the XGBoost valuation engine.


In [ ]:
feat_importances = pd.DataFrame({
    "Feature": all_features,
    "Importance": best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_importances, x="Importance", y="Feature", palette="mako")
plt.title("Top 15 Most Influential Property Valuation Determinants", fontsize=13, fontweight='bold')
plt.xlabel("XGBoost Feature Gain / Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 11. Single Property Live Inference Demonstration
Demonstrating how a single property dictionary is processed through the full end-to-end inference pipeline.


In [ ]:
# Sample custom property
sample_property = {
    "location": "Bellevue Tech Park",
    "sqft_living": 2600,
    "bedrooms": 4,
    "bathrooms": 3.0,
    "sqft_lot": 7500,
    "floors": 2.0,
    "waterfront": 0,
    "view_quality": 2,
    "condition": 4,
    "grade": 9,
    "yr_built": 2016,
    "yr_renovated": 0,
    "has_garage": 1,
    "has_pool": 1,
    "distance_to_center_miles": 5.2
}

# Run through engineered pipeline
sample_df = pd.DataFrame([sample_property])
sample_engineered = engineer_features(sample_df)
sample_trans = preprocessor.transform(sample_engineered[feature_cols])

predicted_val = float(best_xgb.predict(sample_trans)[0])
print("=" * 60)
print("🏡 LIVE PREDICTION RESULT:")
print(f"Property: {sample_property['bedrooms']} Bed, {sample_property['bathrooms']} Bath in {sample_property['location']}")
print(f"Living Area: {sample_property['sqft_living']} sq.ft. | Grade: {sample_property['grade']}/11")
print(f"ESTIMATED VALUATION: ${predicted_val:,.0f}")
print(f"Estimated Price / Sq.Ft: ${predicted_val / sample_property['sqft_living']:.2f}")
print("=" * 60)


## 12. Key Findings & Conclusions
1. **Best Performing Model**: The fine-tuned **XGBoost Regressor** achieved the highest accuracy with an **$R^2$ of 0.9778**, a **Test MAE of $51,514**, and a **MAPE of 4.88%**, outperforming standard Linear Regression and standalone Decision Trees.
2. **Key Valuation Drivers**:
   - `sqft_living` and `grade` (structural architectural quality) are the single largest price determinants.
   - Neighborhood location baseline values and distances to downtown metro significantly modulate the price per square foot.
   - Luxury amenities (waterfront frontage, panoramic views, private pools) command substantial non-linear price premiums.
3. **Deployment**:
   - Web GUI available via `streamlit run app.py`
   - REST API microservice available via `python api.py`
